# Statistical Modeling

A refresher on the *discipline* of building, fitting, checking, and interpreting
statistical models — the umbrella over OLS, GLMs, and their relatives. This notebook
is the conceptual map; the tool-specific notebooks
([statsmodels](statsmodels.ipynb), [lme4](lme4.ipynb),
[mixed-effects-models](mixed-effects-models.ipynb), [pymc](pymc.ipynb)) drill into APIs.

**Domain:** Data Analysis & Research  ·  **runnable:** yes

## 1. What & Why

**Statistical modeling** is the practice of writing down a *generative story* for data —
a small set of equations with parameters — then estimating those parameters, checking
whether the story holds, and using it to **explain** relationships or **predict** new
outcomes *with a quantified uncertainty*.

The core object is almost always a conditional model of an outcome `y` given predictors `X`:

```
y = f(X; β) + noise
```

where `β` are the parameters you estimate and `noise` has an assumed distribution. Linear
regression is `f(X; β) = Xβ` with Gaussian noise; logistic regression keeps `Xβ` but maps
it through a link to a probability; the whole **Generalized Linear Model (GLM)** family is
"a linear predictor `Xβ` + a link function + a noise distribution."

**Why reach for it (instead of just a black-box ML model):**

- You want **interpretable coefficients** — "each extra year of education adds ~$2.4k in
  salary, 95% CI [1.9k, 2.9k]" — not just an accuracy number.
- You need **uncertainty**: standard errors, confidence intervals, p-values, prediction
  intervals. A statistical model is a probability distribution, so it gives you these for free.
- You're doing **inference / science**: testing whether an effect is real, controlling for
  confounders, estimating a treatment effect.
- The dataset is **small-to-medium** and a well-specified parametric model beats a
  data-hungry learner.

**When *not* to:** pure predictive accuracy on large, messy, high-dimensional data where you
don't care about coefficients — gradient boosting or a neural net will usually win. Statistical
modeling trades flexibility for interpretability and honest uncertainty.

## 2. Mental Model

Think of a statistical model as a **dial-and-readout machine** with two halves:

```
        SYSTEMATIC PART                 RANDOM PART
   (what the model predicts)        (what it admits it can't)

   y_hat = g^{-1}( Xβ )      +        a noise distribution
            \______/                    \_______________/
          linear predictor            Normal? Bernoulli? Poisson?
          through a link              -> this is what turns a
                                         point guess into a
                                         probability statement
```

Every regression you know is this picture with different choices:

| Model               | Link `g`        | Noise / family   | Outcome type     |
|---------------------|-----------------|------------------|------------------|
| Linear (OLS)        | identity        | Normal           | continuous       |
| Logistic            | logit           | Bernoulli        | binary           |
| Poisson             | log             | Poisson          | counts           |
| Gamma               | log / inverse   | Gamma            | positive skewed  |

The **fitting** step turns the dials `β` until the model's story is as consistent as
possible with the data — for OLS that's *least squares*, and for GLMs it's *maximum
likelihood* (least squares is the Gaussian special case). The **checking** step asks the
machine to confess: residual plots, dispersion, influence. The **inference** step reads the
uncertainty dials: standard errors → confidence intervals → tests.

Hold this in your head: **linear predictor + link + family + likelihood**. Pick those four
and you've specified the model.

## 3. Key Concepts

- **Linear predictor `η = Xβ`** — a weighted sum of features. "Linear" means linear *in the
  parameters*; you can still include `x²`, `log x`, or interactions as columns of `X`.
- **Link function** — maps the linear predictor to the mean of the outcome (logit for
  probabilities, log for counts). Lets one machinery cover many outcome types.
- **Likelihood** — the probability of the observed data as a function of `β`. **Maximum
  Likelihood Estimation (MLE)** picks the `β` that makes the data most probable. OLS = MLE
  under Gaussian noise.
- **Standard error** — the estimated standard deviation of a coefficient estimate across
  hypothetical repeated samples. The engine of confidence intervals and p-values.
- **Confidence interval** — a range that would contain the true parameter in 95% of repeated
  experiments. Not "95% probability the truth is here" (that's the Bayesian *credible* interval).
- **p-value** — probability of seeing an effect at least this extreme *if the null (effect = 0)
  were true*. Small p = data are surprising under "no effect." It is **not** the probability
  the null is true, and not effect size.
- **R² / deviance** — goodness-of-fit. R² = fraction of variance explained (OLS); deviance
  generalizes it to GLMs. Higher fit ≠ better model (overfitting).
- **AIC / BIC** — information criteria for **model comparison**: fit penalized by parameter
  count. Lower is better; they let you compare non-nested models.
- **Residuals & assumptions** — OLS assumes errors are independent, homoscedastic (constant
  variance), and roughly Normal. Most diagnostics are residual plots checking these.
- **Confounding vs. causation** — a coefficient is the association *holding the other
  predictors fixed*. Causal claims need design (randomization) or causal assumptions, not just
  a good fit.
- **MLE vs. Bayesian** — frequentist MLE gives point estimates + CIs; Bayesian puts a prior on
  `β` and returns a full posterior. See [pymc](pymc.ipynb).

## 4. Setup

The worked examples below use only **NumPy** and **SciPy**, so they run in any standard
scientific-Python kernel with no heavy dependencies. In real work you'd reach for
**statsmodels** (formula API, rich summary tables) or **scikit-learn** (prediction-focused).

```bash
pip install numpy scipy statsmodels scikit-learn pandas
```

In [ ]:
# Only numpy + scipy are needed for everything below — both ship with any
# scientific-Python install. (statsmodels/sklearn shown conceptually in section 7.)
import os
import numpy as np
from scipy import stats, optimize

rng = np.random.default_rng(0)  # reproducible
print("numpy", np.__version__)
print("scipy", __import__("scipy").__version__)

## 5. Worked Examples

Two end-to-end fits, each showing the full loop: **simulate → fit → infer → check**.

1. **Ordinary Least Squares** from the normal equations, with standard errors, t-tests,
   R², and an F-test — the same numbers `statsmodels` prints, computed by hand so the
   machinery is visible.
2. **Logistic regression by maximum likelihood**, fit by numerically maximizing the
   log-likelihood, recovering known coefficients and reading off an odds ratio.

### Example 1 — OLS: fit, inference, and goodness-of-fit

We simulate a known linear world (`y = 2 + 1.5·x1 − 0.8·x2 + noise`), recover the
coefficients via the normal equations `β̂ = (XᵀX)⁻¹ Xᵀy`, and compute the inferential
quantities a regression table reports.

In [ ]:
# --- simulate a known linear world -------------------------------------
n = 200
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
true_beta = np.array([2.0, 1.5, -0.8])      # intercept, x1, x2
sigma = 1.0
X = np.column_stack([np.ones(n), x1, x2])    # design matrix with intercept
y = X @ true_beta + rng.normal(scale=sigma, size=n)

# --- fit by the normal equations ---------------------------------------
XtX_inv = np.linalg.inv(X.T @ X)
beta_hat = XtX_inv @ X.T @ y                  # least-squares estimates

# --- inference: residual variance, std errors, t-stats, p-values -------
resid = y - X @ beta_hat
dof = n - X.shape[1]                          # degrees of freedom
sigma2_hat = resid @ resid / dof              # unbiased error-variance estimate
se = np.sqrt(np.diag(sigma2_hat * XtX_inv))   # standard error of each coef
t_stat = beta_hat / se
p_val = 2 * stats.t.sf(np.abs(t_stat), df=dof)
ci = np.column_stack([beta_hat - stats.t.ppf(0.975, dof) * se,
                      beta_hat + stats.t.ppf(0.975, dof) * se])

names = ["intercept", "x1", "x2"]
print(f"{'term':<10}{'coef':>8}{'std err':>9}{'t':>8}{'p>|t|':>9}"
      f"{'[0.025':>9}{'0.975]':>9}")
for nm, b, s, t, p, lo_hi in zip(names, beta_hat, se, t_stat, p_val, ci):
    print(f"{nm:<10}{b:>8.3f}{s:>9.3f}{t:>8.2f}{p:>9.3g}"
          f"{lo_hi[0]:>9.3f}{lo_hi[1]:>9.3f}")
print(f"\ntrue beta: {true_beta}")

In [ ]:
# --- goodness of fit: R-squared and the overall F-test -----------------
ss_res = resid @ resid
ss_tot = ((y - y.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
adj_r2 = 1 - (ss_res / dof) / (ss_tot / (n - 1))

k = X.shape[1] - 1                            # predictors excluding intercept
f_stat = (r2 / k) / ((1 - r2) / dof)          # H0: all slopes = 0
f_p = stats.f.sf(f_stat, k, dof)

print(f"R-squared      : {r2:.3f}")
print(f"Adj. R-squared : {adj_r2:.3f}")
print(f"F-statistic    : {f_stat:.1f}  (df={k}, {dof})")
print(f"Prob (F-stat)  : {f_p:.3e}")
print("\nReading it: x1 and x2 are both highly significant (tiny p), the CIs")
print("bracket the true 1.5 and -0.8, and the model explains ~", round(r2*100), "% of variance.")

### Example 2 — Logistic regression by maximum likelihood

For a **binary** outcome, OLS is wrong (it can predict probabilities outside [0, 1] and has
non-constant variance). The GLM fix: model `P(y=1) = σ(Xβ)` with the logistic link, and fit
by **maximizing the Bernoulli log-likelihood**. We minimize the negative log-likelihood with
`scipy.optimize` — exactly what a GLM solver does under the hood (it uses Newton/IRLS; we use
a generic optimizer for transparency).

In [ ]:
# --- simulate a binary outcome from a known logistic model -------------
n = 400
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
X = np.column_stack([np.ones(n), x1, x2])
true_beta = np.array([-0.5, 2.0, -1.0])
p_true = 1 / (1 + np.exp(-(X @ true_beta)))
y = rng.binomial(1, p_true)

def neg_log_likelihood(beta):
    eta = X @ beta
    # log(1+e^eta) computed stably via logaddexp(0, eta)
    return np.sum(np.logaddexp(0, eta) - y * eta)

# --- fit: maximize log-likelihood == minimize the negative -------------
res = optimize.minimize(neg_log_likelihood, x0=np.zeros(3), method="BFGS")
beta_hat = res.x

# --- standard errors from the inverse Hessian (observed Fisher info) ----
p_hat = 1 / (1 + np.exp(-(X @ beta_hat)))
W = p_hat * (1 - p_hat)
fisher = X.T @ (W[:, None] * X)               # X' W X
se = np.sqrt(np.diag(np.linalg.inv(fisher)))
z = beta_hat / se
p_val = 2 * stats.norm.sf(np.abs(z))

print(f"{'term':<10}{'coef':>8}{'std err':>9}{'z':>8}{'p>|z|':>10}{'odds ratio':>12}")
for nm, b, s, zz, p in zip(["intercept", "x1", "x2"], beta_hat, se, z, p_val):
    print(f"{nm:<10}{b:>8.3f}{s:>9.3f}{zz:>8.2f}{p:>10.3g}{np.exp(b):>12.3f}")
print(f"\ntrue beta: {true_beta}   converged: {res.success}")

In [ ]:
# --- interpret + evaluate ----------------------------------------------
# Odds ratio for x1 = exp(beta_x1): each +1 in x1 multiplies the ODDS of y=1 by this.
print(f"Odds ratio for x1: {np.exp(beta_hat[1]):.2f}  "
      f"(a one-unit rise in x1 multiplies the odds of y=1 by ~{np.exp(beta_hat[1]):.1f}x)")

# Model fit: McFadden's pseudo-R^2 = 1 - LL(model)/LL(null)
ll_model = -neg_log_likelihood(beta_hat)
p_bar = y.mean()
ll_null = np.sum(y * np.log(p_bar) + (1 - y) * np.log(1 - p_bar))
pseudo_r2 = 1 - ll_model / ll_null
acc = ((p_hat > 0.5).astype(int) == y).mean()
print(f"McFadden pseudo-R^2: {pseudo_r2:.3f}")
print(f"In-sample accuracy : {acc:.3f}")

## 6. Gotchas & Pitfalls

- **p-hacking & the p-value misread.** A p-value is *not* the probability the hypothesis is
  true, and `p < 0.05` is a convention, not a law of nature. Test many things and some cross
  0.05 by chance — pre-register or correct for multiple comparisons.
- **Statistical vs. practical significance.** With huge `n`, trivially small effects get tiny
  p-values. Always report and judge the **effect size** and its CI, not just the star.
- **Assumption violations quietly bias inference.** Heteroscedasticity or correlated errors
  leave coefficients roughly right but make standard errors *wrong* → bogus CIs/p-values.
  Use robust (HC) or clustered standard errors when in doubt.
- **Multicollinearity.** Highly correlated predictors inflate standard errors and make
  individual coefficients unstable/uninterpretable (check VIF). The model can still *predict*
  fine; you just can't trust the per-coefficient story.
- **Correlation ≠ causation.** A controlled-for coefficient is still observational. Omitted
  confounders, reverse causation, and selection bias all masquerade as real effects.
- **Extrapolation.** A model is only trustworthy within the range of `X` it saw. Predicting
  far outside the training support is guesswork dressed as math.
- **Overfitting via flexible specification.** Adding polynomials/interactions always raises
  R². Use **adjusted R²**, AIC/BIC, or held-out data to compare honestly.
- **Linear probability model.** Don't fit OLS to a 0/1 outcome for inference — predicted
  "probabilities" can exceed [0, 1] and the errors are heteroscedastic by construction. Use
  logistic/probit.
- **Separation in logistic regression.** If a predictor perfectly splits the classes, the MLE
  diverges (coefficients → ±∞). Penalize (ridge / Firth) or drop the offender.

## 7. When to Use vs Alternatives

| You want…                                   | Reach for…                                  |
|---------------------------------------------|---------------------------------------------|
| Interpretable coefficients + p-values/CIs   | **statsmodels** (OLS/GLM, formula API, rich `summary()`) |
| Best possible prediction, don't need `β`    | **scikit-learn**, gradient boosting, neural nets |
| Grouped / repeated-measures / hierarchical  | **mixed-effects models** ([lme4](lme4.ipynb), [mixed-effects-models](mixed-effects-models.ipynb)) |
| Full uncertainty, priors, custom likelihood | **Bayesian** modeling ([pymc](pymc.ipynb))  |
| Non-Gaussian outcome (counts, binary, rates)| **GLM** with the right family/link          |
| Nonlinear, high-dim, weak theory            | ML (tree ensembles, deep learning)          |

**Honest trade-offs:**

- **Statistical model vs. ML.** Statistical models give interpretability + calibrated
  uncertainty and excel on small/medium data; ML gives raw predictive power on large, complex
  data but is harder to interpret and doesn't natively quantify parameter uncertainty. They
  overlap — regularized regression is both.
- **Frequentist (MLE) vs. Bayesian.** MLE is fast, needs no priors, and dominates tooling;
  Bayesian gives a full posterior, principled small-sample behavior, and easy uncertainty
  propagation but costs compute and prior choices. For most regressions they agree when data
  are plentiful.
- **OLS vs. GLM.** Use OLS only when the outcome is continuous and roughly Gaussian; otherwise
  pick the GLM family that matches the data-generating process.
- **Build-it-yourself vs. a library.** The from-scratch fits above are for *understanding*. In
  practice use statsmodels/sklearn — they handle numerical stability, robust SEs, missing
  data, and give battle-tested summary tables.

## 8. Resources

- **statsmodels documentation** — the workhorse Python library for OLS/GLM with full inference:
  https://www.statsmodels.org/stable/index.html
- **scikit-learn — Generalized Linear Models** user guide (prediction-focused linear models):
  https://scikit-learn.org/stable/modules/linear_model.html
- **"An Introduction to Statistical Learning" (James, Witten, Hastie, Tibshirani)** — free PDF,
  the canonical accessible treatment of regression and model selection:
  https://www.statlearning.com/
- **OpenIntro Statistics** — free, rigorous-but-readable foundations (inference, regression):
  https://www.openintro.org/book/os/
- **McElreath, "Statistical Rethinking"** (lectures free on YouTube) — the best modern intro to
  the modeling mindset, Bayesian-flavored: https://github.com/rmcelreath/stat_rethinking_2023

**Related notebooks:** [statsmodels](statsmodels.ipynb) ·
[mixed-effects-models](mixed-effects-models.ipynb) · [lme4](lme4.ipynb) ·
[pymc](pymc.ipynb) · [numpy-pandas-scipy](numpy-pandas-scipy.ipynb)

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def ols_fit(x, y):
    """Slope, intercept, variance explained, and the slope's standard error."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE